# Aula 02 — Análise Exploratória de Dados (EDA)

**Objetivo:** conhecer a base de 1.000 vendas antes de criar o PostgreSQL e as camadas Bronze, Silver e Gold.

Execute as células na ordem usando **Shift + Enter**. Cada etapa traz explicação e código comentado.

## 1. Importação e leitura

`import pandas as pd` carrega a biblioteca Pandas e dá a ela o apelido `pd`. `Path` permite localizar o CSV tanto na pasta do projeto quanto na pasta `dados`. `pd.read_csv()` transforma o arquivo em um DataFrame (tabela em memória).

In [ ]:

# Importa a biblioteca Pandas.
# O "pd" é um apelido padrão utilizado para facilitar
# a escrita dos comandos do Pandas.
import pandas as pd

# Mostra a versão instalada do Pandas.
# Isso também confirma que a biblioteca foi carregada corretamente.
print("Versão do Pandas:", pd.__version__)

In [ ]:
import pandas as pd
from pathlib import Path

# Procura o CSV dentro de dados/; se o notebook estiver em notebooks/, usa ../dados/.
caminhos = [Path("dados/vendas.csv"), Path("../dados/vendas.csv")]
caminho_csv = next((p for p in caminhos if p.exists()), None)
if caminho_csv is None:
    raise FileNotFoundError("Não encontrei dados/vendas.csv. Abra o notebook na pasta do projeto.")

# Lê o arquivo e guarda os registros na variável df.
df = pd.read_csv(caminho_csv, encoding="utf-8-sig")
print(f"Arquivo carregado: {caminho_csv}")

## 2. Primeiros registros e dimensões

`head(5)` mostra cinco linhas; `shape` devolve `(linhas, colunas)`; `columns.tolist()` lista os nomes dos campos. Isso ajuda a reconhecer a estrutura da fonte.

In [ ]:
# Exibe uma amostra visual da base.
display(df.head(5))

# Exibe tamanho e nomes das colunas.
print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())

## 3. Tipos de dados

`dtypes` mostra os tipos inferidos; `info()` mostra também a quantidade de valores preenchidos. Datas lidas de CSV geralmente chegam como texto: aqui vamos apenas identificar isso, deixando a conversão para a camada Silver.

In [ ]:
print("Tipos de dados:")
print(df.dtypes)
print("\nResumo:")
df.info()

## 4. Valores ausentes e duplicados

`isna().sum()` conta células vazias por coluna. `duplicated().sum()` conta linhas integralmente repetidas. `duplicated()` aplicado a `id_venda` verifica se uma chave se repete.

In [ ]:
print("Nulos por coluna:")
print(df.isna().sum())
print("\nLinhas totalmente duplicadas:", df.duplicated().sum())
print("IDs de venda duplicados:", df["id_venda"].duplicated().sum())

## 5. Estatísticas e faturamento

`describe()` calcula estatísticas numéricas. `sum()` totaliza `valor_total`. `groupby("categoria")` agrupa vendas por categoria e `sort_values()` ordena do maior faturamento para o menor.

In [ ]:
display(df.describe())

faturamento = df["valor_total"].sum()
print(f"Faturamento total: R$ {faturamento:,.2f}")

faturamento_categoria = (
    df.groupby("categoria", as_index=False)["valor_total"]
      .sum()
      .sort_values("valor_total", ascending=False)
)
display(faturamento_categoria)

## 6. Validações iniciais

Aqui definimos regras simples: 1.000 registros, IDs únicos, nenhum nulo, valores totais não negativos e coerência da fórmula `quantidade × valor_unitario × (1 − desconto/100)`. A tolerância de R$ 0,01 evita reprovação por arredondamentos de centavos. Essas verificações não substituem validações de negócio mais completas.

In [ ]:
recalculado = (
    df["quantidade"] * df["valor_unitario"] *
    (1 - df["desconto_percentual"] / 100)
)

validacoes = {
    "Exatamente 1.000 vendas": len(df) == 1000,
    "IDs únicos": df["id_venda"].is_unique,
    "Nenhum valor ausente": not df.isna().any().any(),
    "Valores não negativos": (df["valor_total"] >= 0).all(),
    "Fórmula de vendas coerente": ((df["valor_total"] - recalculado).abs() <= 0.01).all(),
}

for regra, passou in validacoes.items():
    print(regra, "APROVADO" if passou else "REPROVADO")

print("\nResultado geral:", "APROVADO" if all(validacoes.values()) else "REVISAR")

## 7. Reflexão para os alunos

1. Por que devemos analisar os tipos antes de criar tabelas no banco?
2. Qual a diferença entre uma linha duplicada e um ID duplicado?
3. Uma base sem nulos está necessariamente correta?
4. O que deve acontecer com registros inválidos na camada Silver?

**Próxima aula:** criar o PostgreSQL com Docker e carregar a camada Bronze.